# 2. Evaluación de Sistemas RAG con LangSmith

## Objetivos de Aprendizaje
- Comprender la importancia de la evaluación en sistemas RAG.
- Configurar LangSmith para trazabilidad y evaluación.
- Crear un dataset de evaluación con preguntas y respuestas de referencia.
- Escribir **evaluadores personalizados** para métricas de calidad.
- Analizar los resultados de la evaluación para optimizar el sistema.

## ¿Qué es LangSmith?

LangSmith es una plataforma de LangChain para la observabilidad, el monitoreo y la evaluación de aplicaciones construidas con LLMs. Permite visualizar cada paso de una cadena o agente, analizar su rendimiento y evaluar la calidad de las respuestas de forma sistemática.

Para un sistema RAG, LangSmith nos ayuda a responder preguntas clave:
- **Recuperación (Retrieval)**: ¿Los documentos que encontramos son relevantes para la pregunta?
- **Generación (Generation)**: ¿La respuesta generada es fiel a los documentos recuperados? ¿Responde correctamente a la pregunta del usuario?

## ⚠️ Arquitectura Híbrida

Este notebook usa **dos proveedores** (igual que el resto del curso):

| Componente | Proveedor | Modelo |
| :--- | :--- | :--- |
| **Chat/Generación** | OpenRouter | `minimax/minimax-m3:free` |
| **Evaluador** | OpenRouter | `minimax/minimax-m3:free` |
| **Trazabilidad** | LangSmith | (gratis para uso personal) |

## 📌 Nota sobre el bug del evaluador `qa`

En versiones anteriores se usaba `load_evaluator("qa")` de LangChain. **Está deprecado** y su interfaz no coincide con cómo LangSmith pasa los datos a los evaluadores. Por eso lo reemplazamos con **evaluadores personalizados** que reciben `run` y `example` y devuelven una puntuación.

In [ ]:
# Instalación de dependencias
# - langsmith: cliente para trazabilidad y evaluación
# - langchain-openai: para ChatOpenAI (apuntando a OpenRouter)
# - python-dotenv: carga de variables .env
!pip install langsmith langchain-openai openai python-dotenv -q
print("✅ Dependencias instaladas")

In [ ]:
import os
from dotenv import load_dotenv
from getpass import getpass
import json

# Cargar variables desde .env
load_dotenv()

# --- Validación de claves ---

# 1. Clave de OpenRouter (para chat y evaluación)
OPENROUTER_KEY = os.environ.get("OPENROUTER_API_KEY")
if not OPENROUTER_KEY:
    print("🔑 No se encontró OPENROUTER_API_KEY.")
    OPENROUTER_KEY = getpass("Ingresa tu clave de OpenRouter: ")
    os.environ["OPENROUTER_API_KEY"] = OPENROUTER_KEY

if not OPENROUTER_KEY or len(OPENROUTER_KEY) < 20:
    raise ValueError("❌ OPENROUTER_API_KEY inválida.")

# 2. Clave de LangSmith (para trazabilidad)
LANGCHAIN_KEY = os.environ.get("LANGCHAIN_API_KEY")
if not LANGCHAIN_KEY:
    print("🔑 No se encontró LANGCHAIN_API_KEY.")
    print("   Obtén tu clave gratis en https://smith.langchain.com")
    LANGCHAIN_KEY = getpass("Ingresa tu clave de LangSmith (lsv2_...): ")
    os.environ["LANGCHAIN_API_KEY"] = LANGCHAIN_KEY

# Configurar las variables de LangSmith
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = os.environ.get("LANGCHAIN_PROJECT", "RAG-Evaluation-Curso")

print("✅ Claves configuradas:")
print(f"   OPENROUTER_API_KEY: {OPENROUTER_KEY[:12]}...")
print(f"   LANGCHAIN_API_KEY: {LANGCHAIN_KEY[:10]}...")
print(f"   LANGCHAIN_PROJECT: {os.environ['LANGCHAIN_PROJECT']}")

In [ ]:
from langsmith import Client

# Inicializar el cliente de LangSmith
try:
    client = Client()
    print("✅ Cliente de LangSmith inicializado correctamente.")
except Exception as e:
    print(f"❌ Error inicializando LangSmith: {e}")
    print("   Verifica que tu LANGCHAIN_API_KEY sea válida.")
    raise

## 2. Sistema RAG Básico

Reutilizamos un RAG simple con las correcciones aplicadas:
- **Filtro de stopwords** en la recuperación (evita falsos positivos).
- **Separador correcto** entre documentos en el contexto.

In [ ]:
from openai import OpenAI

# --- Configuración del cliente (OpenRouter) ---
MODELO = "minimax/minimax-m3:free"
BASE_URL = "https://openrouter.ai/api/v1"

openai_client = OpenAI(base_url=BASE_URL, api_key=OPENROUTER_KEY)

# Headers opcionales para identificar tu app
DEFAULT_HEADERS = {
    "HTTP-Referer": "http://localhost:8000",
    "X-Title": "RAG Evaluation",
}

# --- Base de documentos ---
documents = [
    "La inteligencia artificial es una rama de la informática que busca crear máquinas capaces de realizar tareas que requieren inteligencia humana.",
    "Los modelos de lenguaje grande (LLM) son sistemas de IA entrenados en enormes cantidades de texto para generar y comprender lenguaje natural.",
    "RAG (Retrieval-Augmented Generation) combina la búsqueda de información relevante con la generación de texto para producir respuestas más precisas.",
    "LangChain es un framework que facilita el desarrollo de aplicaciones con modelos de lenguaje, proporcionando herramientas para cadenas y agentes.",
    "El prompt engineering es la práctica de diseñar instrucciones efectivas para obtener los mejores resultados de los modelos de IA."
]

print(f"📚 Base de datos con {len(documents)} documentos cargada.")
print(f"✅ Cliente OpenRouter inicializado con modelo '{MODELO}'.")

In [ ]:
from langsmith.run_helpers import traceable
import openai

# --- Stopwords en español ---
STOPWORDS_ES = {
    "de", "la", "que", "el", "en", "y", "a", "los", "se", "del",
    "las", "un", "por", "con", "no", "una", "su", "para", "es",
    "al", "lo", "como", "más", "o", "pero", "sus", "le", "ha",
    "me", "si", "sin", "sobre", "este", "ya", "entre", "cuando",
    "qué", "cómo", "cuál", "quién", "dónde", "cuándo",
}

def extraer_palabras_clave(texto):
    """Filtra stopwords y palabras cortas."""
    texto = texto.lower()
    for signo in "¿?¡!.,;:()[]{}":
        texto = texto.replace(signo, " ")
    return [p for p in texto.split() if p not in STOPWORDS_ES and len(p) >= 3]


@traceable(name="Recuperacion de Documentos")
def simple_retrieval(query, documents):
    """Recuperación por palabras clave (con filtro de stopwords)."""
    query_words = set(extraer_palabras_clave(query))
    if not query_words:
        return []
    
    # Puntuar cada documento por número de coincidencias
    scored = []
    for doc in documents:
        doc_words = set(extraer_palabras_clave(doc))
        overlap = len(query_words.intersection(doc_words))
        if overlap > 0:
            scored.append((overlap, doc))
    
    scored.sort(key=lambda x: x[0], reverse=True)
    return [doc for _, doc in scored[:3]]


@traceable(name="Generacion de Respuesta")
def generate_response(client, query, context):
    """Genera la respuesta usando OpenRouter."""
    prompt = f"""Contexto:
{context}

Pregunta: {query}

Responde basándote únicamente en el contexto proporcionado."""
    
    try:
        response = client.chat.completions.create(
            model=MODELO,
            messages=[{"role": "user", "content": prompt}],
            temperature=0.1,
            max_tokens=300,
            extra_headers=DEFAULT_HEADERS,
        )
        return response.choices[0].message.content
    except openai.AuthenticationError:
        return "Error: autenticación fallida"
    except openai.RateLimitError:
        return "Error: rate limit alcanzado"
    except Exception as e:
        return f"Error: {e}"


@traceable(name="Pipeline RAG Completo")
def rag_pipeline(query):
    """Pipeline RAG completo: retrieval + generation."""
    context_docs = simple_retrieval(query, documents)
    # ✅ CORREGIDO: separador '\n\n' entre documentos
    context = "\n\n".join(context_docs)
    answer = generate_response(openai_client, query, context)
    return {"answer": answer, "context": context_docs}


print("✅ Funciones del pipeline RAG definidas y trazables")

In [ ]:
# Prueba del pipeline
resultado = rag_pipeline("¿Qué es RAG?")
print("=== Respuesta del RAG ===")
print(resultado["answer"])
print(f"\n📋 Documentos recuperados: {len(resultado['context'])}")

## 3. Creación de un Dataset de Evaluación

Para evaluar nuestro sistema, necesitamos un "ground truth": un conjunto de preguntas y las respuestas que consideramos correctas.

In [ ]:
dataset_name = "Dataset RAG Básico - Fundamentos IA"
description = "Preguntas y respuestas sobre conceptos básicos de IA para evaluar un RAG simple."

# Eliminar dataset si ya existe para evitar duplicados
try:
    existing_dataset = client.read_dataset(dataset_name=dataset_name)
    client.delete_dataset(dataset_id=str(existing_dataset.id))
    print(f"🗑️ Dataset '{dataset_name}' existente eliminado.")
except Exception:
    pass

# Crear el nuevo dataset
dataset = client.create_dataset(
    dataset_name=dataset_name,
    description=description,
)

print(f"✅ Dataset '{dataset_name}' creado.")

In [ ]:
# Añadir ejemplos al dataset
ejemplos = [
    {
        "inputs": {"query": "¿Qué es la inteligencia artificial?"},
        "outputs": {"answer": "La inteligencia artificial es una rama de la informática que busca crear máquinas capaces de realizar tareas que requieren inteligencia humana."}
    },
    {
        "inputs": {"query": "¿Para qué sirve LangChain?"},
        "outputs": {"answer": "LangChain es un framework que facilita el desarrollo de aplicaciones con modelos de lenguaje, proporcionando herramientas para cadenas y agentes."}
    },
    {
        "inputs": {"query": "Explica qué es RAG"},
        "outputs": {"answer": "RAG (Retrieval-Augmented Generation) combina la búsqueda de información relevante con la generación de texto para producir respuestas más precisas."}
    },
]

for ej in ejemplos:
    client.create_example(
        inputs=ej["inputs"],
        outputs=ej["outputs"],
        dataset_id=dataset.id,
    )

print(f"✅ {len(ejemplos)} ejemplos añadidos al dataset '{dataset_name}'.")

## 4. Evaluadores Personalizados

En lugar de usar `load_evaluator("qa")` (deprecado y con interfaz incompatible), definimos **evaluadores personalizados**. Cada evaluador:

- Recibe un `run` (la ejecución del pipeline) y un `example` (el ejemplo del dataset).
- Devuelve un diccionario con `key` (nombre de la métrica), `score` (0-1) y opcionalmente `comment`.

Así LangSmith entiende perfectamente qué datos usar y no falla con `Missing some input keys`.

In [ ]:
@traceable(name="Evaluator - Correctness")
def correctness_evaluator(run, example):
    """Evalúa si la respuesta generada es correcta comparándola con la referencia."""
    question = example.inputs.get("query", "")
    prediction = run.outputs.get("answer", "")
    reference = example.outputs.get("answer", "")
    
    eval_prompt = f"""Evalúa si la respuesta generada es correcta comparándola con la respuesta de referencia.

Pregunta: {question}

Respuesta generada: {prediction}

Respuesta de referencia: {reference}

¿La respuesta generada es correcta y equivalente a la de referencia?
Responde SOLO con un número del 0 al 1 (donde 1 = completamente correcta, 0 = incorrecta):"""
    
    try:
        response = openai_client.chat.completions.create(
            model=MODELO,
            messages=[{"role": "user", "content": eval_prompt}],
            temperature=0.0,
            max_tokens=10,
            extra_headers=DEFAULT_HEADERS,
        )
        score = float(response.choices[0].message.content.strip())
        score = max(0.0, min(1.0, score))  # Asegurar rango [0, 1]
    except Exception as e:
        score = 0.0
    
    return {"key": "correctness", "score": score}


@traceable(name="Evaluator - Faithfulness")
def faithfulness_evaluator(run, example):
    """Evalúa si la respuesta es fiel al contexto recuperado (no alucina)."""
    prediction = run.outputs.get("answer", "")
    context_docs = run.outputs.get("context", [])
    context = "\n\n".join(context_docs) if isinstance(context_docs, list) else str(context_docs)
    
    if not context:
        return {"key": "faithfulness", "score": 0.0, "comment": "Sin contexto"}
    
    eval_prompt = f"""Evalúa si la respuesta está basada ÚNICAMENTE en el contexto proporcionado.

Contexto:
{context}

Respuesta generada:
{prediction}

¿La respuesta está completamente basada en el contexto?
Responde SOLO con un número del 0 al 1 (donde 1 = totalmente fiel, 0 = alucina):"""
    
    try:
        response = openai_client.chat.completions.create(
            model=MODELO,
            messages=[{"role": "user", "content": eval_prompt}],
            temperature=0.0,
            max_tokens=10,
            extra_headers=DEFAULT_HEADERS,
        )
        score = float(response.choices[0].message.content.strip())
        score = max(0.0, min(1.0, score))
    except Exception:
        score = 0.0
    
    return {"key": "faithfulness", "score": score}


@traceable(name="Evaluator - Retrieval Quality")
def retrieval_evaluator(run, example):
    """Evalúa si se recuperaron documentos relevantes."""
    context_docs = run.outputs.get("context", [])
    
    if not context_docs:
        return {"key": "retrieval_score", "score": 0.0, "comment": "Sin documentos"}
    
    # Métrica simple: al menos 1 documento recuperado
    score = min(1.0, len(context_docs) / 3.0)
    
    return {"key": "retrieval_score", "score": score, "comment": f"{len(context_docs)} docs"}


print("✅ Evaluadores personalizados definidos:")
print("   • correctness_evaluator")
print("   • faithfulness_evaluator")
print("   • retrieval_evaluator")

## 5. Ejecución de la Evaluación

Ahora ejecutamos la evaluación con LangSmith. Le pasamos:

- **`target_function`**: la función a evaluar (nuestro RAG).
- **`data`**: el nombre del dataset en LangSmith.
- **`evaluators`**: la lista de evaluadores personalizados.

In [ ]:
from langsmith.evaluation import evaluate

# Función que será evaluada
def target_function(inputs):
    """Envuelve el rag_pipeline para que LangSmith lo llame."""
    return rag_pipeline(inputs["query"])

# Ejecutar la evaluación
experiment_results = evaluate(
    target_function,                             # Función a evaluar
    data=dataset_name,                           # Dataset en LangSmith
    evaluators=[                                 # Evaluadores personalizados
        correctness_evaluator,
        faithfulness_evaluator,
        retrieval_evaluator,
    ],
    experiment_prefix="RAG-Basico-Primera-Prueba",  # Prefijo del experimento
    metadata={"version": "1.0.0", "modelo": MODELO},
)

print("✅ Evaluación completada. Revisa los resultados en LangSmith.")

## 6. Análisis de Resultados

Una vez completada la evaluación, puedes navegar a la pestaña **Experiments** en tu proyecto de LangSmith.

Allí encontrarás:
1. **Resumen del experimento** con las puntuaciones medias de cada métrica.
2. **Tabla detallada** con cada pregunta del dataset, la respuesta generada, la respuesta de referencia y las puntuaciones.
3. **Traza completa** al hacer clic en cada fila (qué documentos se recuperaron, qué prompt se usó, etc.).

### ¿Cómo interpretar las métricas?

| Métrica | Qué mide | Cómo mejorarla |
| :--- | :--- | :--- |
| **correctness** | ¿La respuesta es correcta vs. referencia? | Mejorar el prompt o el retrieval |
| **faithfulness** | ¿La respuesta se basa en el contexto? | Reducir `temperature`, mejorar el prompt |
| **retrieval_score** | ¿Se recuperaron docs relevantes? | Mejorar chunking o usar búsqueda semántica |

### Próximos pasos

- **Iterar el prompt**: si `correctness` es bajo pero `faithfulness` es alto, el problema es el retrieval.
- **Cambiar el retrieval**: si ambos son bajos, migrar a búsqueda semántica con embeddings.
- **Comparar experimentos**: LangSmith permite comparar múltiples ejecuciones (v1 vs v2 del prompt).

## Conclusión

La evaluación es un pilar fundamental en el desarrollo de sistemas de IA robustos. LangSmith nos ofrece un conjunto de herramientas poderosas para automatizar este proceso en sistemas RAG, permitiéndonos pasar de un desarrollo basado en la intuición a uno guiado por datos y métricas objetivas.